In [1]:
!pip install -q datasets soundfile librosa

In [2]:
import os
import json
import wave
import zipfile
import numpy as np
import soundfile as sf
import librosa
from datasets import load_dataset, Audio
# ================= CẤU HÌNH 3 NGUỒN (MỖI NGUỒN 2 FILE) =================
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
SOURCES = [
    {
        "id": "gpt-omni/VoiceAssistant-400K",
        "split": "train",
        "data_files": "data/train-00000-of-00325.parquet",
        "tag": "voiceassistant",
        "n": 2,
    },
    {"id": "AudioLLMs/spoken_squad_test", "split": "test", "tag": "spokensquad", "n": 2},
    {"id": "AudioLLMs/alpaca_audio_test", "split": "test", "tag": "alpacaaudio", "n": 2},
]
TARGET_SR = 16000
MIN_SEC   = 0.5
MAX_SEC   = 30.0
OUT_DIR   = "/kaggle/working/wav_extra"
START_IDX = 15  # Nối tiếp sau 14.wav
# =====================================================================
os.makedirs(OUT_DIR, exist_ok=True)

In [5]:
import io
import os
import glob
import json
import wave
import zipfile
import numpy as np
import soundfile as sf
import librosa
from datasets import load_dataset, Audio

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

SOURCES = [
    {
        "id": "gpt-omni/VoiceAssistant-400K",
        "split": "train",
        "data_files": "data/train-00000-of-00325.parquet",
        "tag": "voiceassistant",
        "n": 3,
    },
    {
        "id": "AudioLLMs/alpaca_audio_test",
        "split": "test",
        "tag": "alpacaaudio",
        "n": 3,
    },
]
TARGET_SR = 16000
MIN_SEC   = 0.5
MAX_SEC   = 30.0
OUT_DIR   = "/kaggle/working/wav_extra"
START_IDX = 15

os.makedirs(OUT_DIR, exist_ok=True)
# Xóa các file .wav cũ của lần chạy trước để không bị lẫn spokensquad
for old_f in glob.glob(os.path.join(OUT_DIR, "*.wav")):
    os.remove(old_f)


def verify_wav_file(path: str, expected_sr: int = 16000):
    if not os.path.isfile(path) or os.path.getsize(path) < 44:
        return False, "Missing or < 44 bytes"
    with open(path, "rb") as f:
        hdr = f.read(12)
    if hdr[:4] != b"RIFF" or hdr[8:12] != b"WAVE":
        return False, f"Invalid RIFF/WAVE header: {hdr!r}"
    try:
        with wave.open(path, "rb") as wf:
            ch, sw, sr, nf, comp = (
                wf.getnchannels(), wf.getsampwidth(),
                wf.getframerate(), wf.getnframes(), wf.getcomptype()
            )
            if comp != "NONE" or ch != 1 or sw != 2 or sr != expected_sr or nf <= 0:
                return False, f"Bad format: ch={ch}, sw={sw}, sr={sr}, nf={nf}, comp={comp}"
            if len(wf.readframes(nf)) != nf * ch * sw:
                return False, "Truncated frames"
            return True, f"Valid WAV | {sr}Hz | 1ch | 16-bit PCM | {nf/sr:.2f}s"
    except Exception as e:
        return False, f"wave.open error: {e}"


def decode_audio_entry(sample: dict):
    """Lấy bytes hoặc array và đúng cột text câu hỏi (ưu tiên speech_instruction)."""
    audio_obj = None
    for k in ("question_audio", "audio", "speech", "input_audio", "context"):
        v = sample.get(k)
        if v is not None and (isinstance(v, dict) or hasattr(v, "get_all_samples")):
            audio_obj = v
            break
    if audio_obj is None:
        for v in sample.values():
            if isinstance(v, dict) and ("bytes" in v or "array" in v):
                audio_obj = v
                break
    if audio_obj is None:
        return None, None, ""

    # Ưu tiên speech_instruction (cho alpaca_audio_test) rồi mới tới question
    text_val = ""
    for k in ("speech_instruction", "question", "instruction", "prompt", "text", "query"):
        v = sample.get(k)
        if isinstance(v, str) and v.strip():
            text_val = " ".join(v.strip().split())
            break

    if isinstance(audio_obj, dict):
        if audio_obj.get("bytes"):
            raw = io.BytesIO(audio_obj["bytes"])
            try:
                arr, sr = sf.read(raw, dtype="float32")
            except Exception:
                raw.seek(0)
                arr, sr = librosa.load(raw, sr=None)
            return np.asarray(arr, dtype=np.float32), int(sr), text_val
        elif "array" in audio_obj and "sampling_rate" in audio_obj:
            return np.asarray(audio_obj["array"], dtype=np.float32), int(audio_obj["sampling_rate"]), text_val
    elif hasattr(audio_obj, "get_all_samples"):
        s = audio_obj.get_all_samples()
        return s.data.squeeze(0).numpy().astype(np.float32), int(s.sample_rate), text_val

    return None, None, text_val


saved = []
file_idx = START_IDX

for src in SOURCES:
    print(f"\n--- Streaming {src['id']} (split={src['split']}, target={src['n']} files) ---")
    ds = load_dataset(
        src["id"],
        split=src["split"],
        data_files=src.get("data_files"),
        streaming=True,
    )
    for col in ("question_audio", "audio", "speech", "input_audio", "context"):
        if ds.features and col in ds.features and isinstance(ds.features[col], Audio):
            ds = ds.cast_column(col, Audio(decode=False))

    got = 0
    for idx, sample in enumerate(ds):
        if got >= src["n"]:
            break

        arr, sr, q_text = decode_audio_entry(sample)
        if arr is None:
            continue

        if arr.ndim > 1:
            arr = np.mean(arr, axis=-1 if arr.shape[-1] <= 8 else 0)
        if sr != TARGET_SR:
            arr = librosa.resample(arr, orig_sr=sr, target_sr=TARGET_SR)

        dur = len(arr) / float(TARGET_SR)
        peak = float(np.max(np.abs(arr))) if len(arr) else 0.0
        if dur < MIN_SEC or dur > MAX_SEC or peak < 1e-4:
            continue
        if peak > 0.99:
            arr = arr * (0.95 / peak)

        out_name = f"{file_idx:02d}_{src['tag']}.wav"
        out_path = os.path.join(OUT_DIR, out_name)
        sf.write(out_path, arr, TARGET_SR, subtype="PCM_16", format="WAV")

        ok, msg = verify_wav_file(out_path, expected_sr=TARGET_SR)
        if not ok:
            print(f"  [FAIL] {out_name}: {msg}")
            os.remove(out_path)
            continue

        print(f"  [OK] {out_name} | {msg}")
        saved.append({
            "file": out_name,
            "source": src["id"],
            "duration_s": round(dur, 2),
            "question": q_text,
            "verify": msg,
        })
        got += 1
        file_idx += 1

meta_path = os.path.join(OUT_DIR, "metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(saved, f, indent=2, ensure_ascii=False)

zip_path = "/kaggle/working/wav_extra_6.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for item in saved:
        zf.write(os.path.join(OUT_DIR, item["file"]), arcname=item["file"])
    zf.write(meta_path, arcname="metadata.json")

print("\n" + "=" * 80)
print("QUESTION LIST FOR THE 6 NEW WAV FILES")
print("=" * 80)
for item in saved:
    print(f"WAV     : {item['file']} ({item['duration_s']}s)")
    print(f"Question: {item['question']}")
    print("-" * 80)

print(f"\nDONE! Saved {len(saved)}/6 verified .wav files -> {OUT_DIR}")
print(f"Zip file ready at: {zip_path}")


--- Streaming gpt-omni/VoiceAssistant-400K (split=train, target=3 files) ---
  [OK] 15_voiceassistant.wav | Valid WAV | 16000Hz | 1ch | 16-bit PCM | 3.01s
  [OK] 16_voiceassistant.wav | Valid WAV | 16000Hz | 1ch | 16-bit PCM | 3.56s
  [OK] 17_voiceassistant.wav | Valid WAV | 16000Hz | 1ch | 16-bit PCM | 3.17s

--- Streaming AudioLLMs/alpaca_audio_test (split=test, target=3 files) ---
  [OK] 18_alpacaaudio.wav | Valid WAV | 16000Hz | 1ch | 16-bit PCM | 3.01s
  [OK] 19_alpacaaudio.wav | Valid WAV | 16000Hz | 1ch | 16-bit PCM | 3.80s
  [OK] 20_alpacaaudio.wav | Valid WAV | 16000Hz | 1ch | 16-bit PCM | 5.47s

QUESTION LIST FOR THE 6 NEW WAV FILES
WAV     : 15_voiceassistant.wav (3.01s)
Question: What makes you different from other assistant models?
--------------------------------------------------------------------------------
WAV     : 16_voiceassistant.wav (3.56s)
Question: Can you share a brief summary of your existence?
----------------------------------------------------------------